In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]  # adjust based on output
SRC_PATH = PROJECT_ROOT / "src"

sys.path.insert(0, str(SRC_PATH))

from sephora_sentiment import show_dashboard, show_overview 

In [2]:
from sephora_sentiment import _load_data, _data
import pandas as pd

_load_data()

brand_health = _data["brand_agg"].copy()
brand_health["health_score"] = (
    brand_health["avg_sentiment"] * 0.3
    + brand_health["rating_sentiment_alignment"] * 0.25
    + brand_health["sentiment_polarization"] * 0.25
    + (1 - brand_health["pct_1_star"]) * 0.2
)
brand_health = brand_health.sort_values("health_score", ascending=False).reset_index(drop=True)
brand_health.index += 1
brand_health.index.name = "Rank"

display_cols = ["brand", "health_score", "avg_sentiment", "avg_rating",
                "total_mismatch_rate", "n_reviews", 
                "complaint_concentration_label", "value_driver_label"]

pd.set_option("display.max_rows", None)
brand_health[display_cols].style.format({
    "health_score": "{:.3f}",
    "avg_sentiment": "{:.3f}",
    "avg_rating": "{:.2f}",
    "total_mismatch_rate": "{:.1%}",
    "n_reviews": "{:,.0f}",
}).background_gradient(subset=["health_score"], cmap="RdYlGn")

Loading dashboard data from CSVs...
  Loading reviews (subset columns)...
  Indexing by brand...
  Loaded 304 brands, 227,783 reviews in 0.4s


,brand,health_score,avg_sentiment,avg_rating,total_mismatch_rate,n_reviews,complaint_concentration_label,value_driver_label
Rank,,,,,,,,
1,Harlem Perfume Co.,0.979,0.965,4.79,3.3%,582,insufficient_data,price_neutral
2,Cyklar,0.970,0.911,4.91,1.2%,80,insufficient_data,price_neutral
3,fel beauty,0.967,0.941,4.68,6.3%,111,insufficient_data,price_neutral
4,Miu Miu,0.964,0.922,4.75,5.0%,100,insufficient_data,price_neutral
5,Unove,0.963,0.896,4.82,2.2%,45,insufficient_data,price_neutral
6,Erborian,0.956,0.870,4.87,1.0%,100,insufficient_data,perceived_good_value
7,Katini Skin,0.953,0.871,4.81,1.9%,312,insufficient_data,perceived_good_value
8,Azzaro,0.950,0.891,4.77,4.9%,369,concentrated,perceived_good_value
9,Hugo Boss,0.948,0.895,4.61,7.3%,218,insufficient_data,perceived_good_value


In [3]:
show_overview()

In [4]:
from sephora_sentiment import write_all_brand_htmls
write_all_brand_htmls()

Writing 304 brand dashboards to /Users/hkbuttar/Downloads/Algorithmic_Marketing_Final_Project/notebooks/independent/outputs/brand_dashboards/
  -> /Users/hkbuttar/Downloads/Algorithmic_Marketing_Final_Project/notebooks/independent/outputs/brand_dashboards/PAT_McGRATH_LABS.html
  -> /Users/hkbuttar/Downloads/Algorithmic_Marketing_Final_Project/notebooks/independent/outputs/brand_dashboards/Urban_Decay.html
  -> /Users/hkbuttar/Downloads/Algorithmic_Marketing_Final_Project/notebooks/independent/outputs/brand_dashboards/Yves_Saint_Laurent.html
  -> /Users/hkbuttar/Downloads/Algorithmic_Marketing_Final_Project/notebooks/independent/outputs/brand_dashboards/Sol_de_Janeiro.html
  -> /Users/hkbuttar/Downloads/Algorithmic_Marketing_Final_Project/notebooks/independent/outputs/brand_dashboards/Olaplex.html
  -> /Users/hkbuttar/Downloads/Algorithmic_Marketing_Final_Project/notebooks/independent/outputs/brand_dashboards/Murad.html
  -> /Users/hkbuttar/Downloads/Algorithmic_Marketing_Final_Project/

PosixPath('/Users/hkbuttar/Downloads/Algorithmic_Marketing_Final_Project/notebooks/independent/outputs/brand_dashboards_index.html')

In [5]:
show_dashboard()

Dropdown(description='Brand:', layout=Layout(width='450px'), options=('PAT McGRATH LABS', 'Urban Decay', 'Yves…

Output()

## Sephora Brand Health Overview

### Composite Health Score and Overall Distribution

The brand health score synthesizes average sentiment, average star rating, and mismatch rate — the proportion of reviews where written sentiment and numerical rating diverge directionally — into a single composite index ranging from 0 to 1. Across the 304 ranked brands, scores span from 0.979 at the top (Harlem Perfume Co.) down to the low 0.600s at the bottom of the visible distribution, with the bulk of the catalog concentrated between 0.85 and 0.93. The distribution is left-skewed — most brands cluster in a relatively healthy band, with a thin tail of genuinely distressed brands pulling the lower end. This compression at the top reflects a selection effect: Sephora's curation model filters out the lowest-quality products before they accumulate enough review volume to appear in this analysis, meaning the floor for a brand with meaningful review volume is already fairly high.

### The Sentiment-Rating Relationship and Mismatch as a Third Dimension

The scatter plot maps each brand on two axes — average VADER compound sentiment (x) and average star rating (y) — with bubble size encoding review volume and color encoding mismatch rate on a green-to-red scale. The dominant pattern is a positive diagonal relationship running from the lower-left (low sentiment, low rating) to the upper-right (high sentiment, high rating), confirming that sentiment and rating are generally coupled. The main cluster of high-volume brands sits between 0.70 and 0.90 sentiment and between 4.3 and 4.7 stars — the operational center of Sephora's prestige catalog.

The color gradient overlaid on this diagonal is the most analytically rich feature of the chart. The upper-right cluster — high sentiment, high rating, large bubbles — is predominantly deep green, indicating very low mismatch rates. These are brands whose consumers rate numerically in alignment with what they write: when they say positive things, they give high stars, and when they give high stars, the language matches. The top-ranked brands in the table confirm this: Harlem Perfume Co. (health score 0.979, mismatch rate 3.3%), Erborian (0.956, 1.0%), Katini Skin (0.953, 1.9%), Azzaro (0.950, 4.9%), and Hugo Boss (0.948, 7.3%) all combine strong sentiment, solid ratings, and low mismatch. These are brands with clear, consistent consumer experiences that generate coherent signals across both measurement dimensions.

As the chart moves toward the lower-left, colors shift through yellow and orange into red, indicating rising mismatch rates alongside declining sentiment and rating. The large orange and red bubbles in the 0.50–0.65 sentiment range represent brands with meaningful review volume but significant internal inconsistency — consumers who write moderately positive text while giving low stars, or who give acceptable stars while writing critically in the text body. High mismatch rates are a diagnostic indicator of expectation management problems: they arise when a portion of the consumer base is satisfying the social norm of leaving a review without feeling the experience fully justified the rating they gave, or when product-level variation within a brand's catalog produces wildly different experiences that average into a mediocre middle.

### Notable Brand Positions

Several specific brand positions in the table and scatter are worth calling out. GOOP (rank 97, health score 0.873, mismatch rate 14.7%) has the highest mismatch rate among brands in the visible top-100, reflecting its well-documented positioning tension — a lifestyle brand with premium pricing whose beauty products attract consumers with high expectations that are inconsistently met, generating the sentiment-rating inconsistency characteristic of overpromising brands. TOM FORD (rank 41, health score 0.911, mismatch rate 11.6%) and Jo Malone London (rank 42, health score 0.910, mismatch rate 11.1%) both sit in the moderate-to-high mismatch range despite strong health scores, reflecting that luxury fragrance brands attract a bifurcated reviewer population: devoted loyalists who write and rate effusively alongside newcomers whose first-purchase experience of a $200+ fragrance does not always meet expectations set by aspirational brand positioning.

The value driver labels in the table add another layer of interpretation. The perceived_good_value classification — assigned to Erborian, Katini Skin, Hugo Boss, Hanyul, banu, Ralph Lauren, Dyson, and others — identifies brands where positive price mentions in reviews exceed negative ones, suggesting that consumers believe they are getting more than they paid for relative to alternatives. These brands tend to occupy the upper-left of the scatter relative to their price tier peers: better ratings and sentiment than a consumer would expect given the price point. The price_driven_negativity classification — assigned to rhode, Kaja, ALPYN, Bobbi Brown, Gisou, LANEIGE, LoveShackFancy, and others — identifies brands where negative price mentions are disproportionately present in critical reviews, meaning that value-for-money dissatisfaction is driving a portion of the mismatch and negative sentiment. These brands cluster in the mid-sentiment, moderate-mismatch zone of the scatter — not the worst performers, but brands where pricing is actively working against consumer satisfaction rather than being ignored.

### Structural Observations Across the Health Ranking

Several patterns emerge from reading across the full ranked table. Fragrance brands occupy a disproportionate share of the top 50 health ranks — Harlem Perfume Co., Miu Miu, Azzaro, Hugo Boss, Montblanc, Jimmy Choo, Boy Smells, Jean Paul Gaultier, Henry Rose, Valentino, Rabanne, and Gucci all appear in the top 50. Fragrance is a category where expectation management is structurally easier: consumers buy fragrance based on sensory experience in-store or on gifting occasions, and the primary failure mode — not liking the scent — generates straightforward low ratings with sentiment that matches, keeping mismatch rates low. Fragrance also benefits from low ingredient-claim complexity; unlike skincare or haircare, there are no efficacy promises about acne clearance or hair repair that can fail to materialize and generate the disappointed-but-rating-politely dynamic that inflates mismatch.

Professional and specialist tool brands also perform well in the upper health ranks — T3 (rank 56), Dyson (rank 78), BaBylissPRO (rank 96), Shark Beauty (rank 94) — consistent with the finding that objectively evaluable performance categories generate lower mismatch and more coherent sentiment-rating coupling. Consumers who buy a hair dryer can evaluate whether it dries faster or smoother than their previous tool, generating review language that directly corresponds to their numerical rating.

The brands at the lower end of the visible health distribution — those in the yellow-to-orange zone of the color scale with health scores in the 0.86–0.87 range — tend to be either high-complexity functional skincare brands (StriVectin, Dr. Barbara Sturm, NuFACE) where consumer results vary substantially by skin type and usage consistency, or high-volume color makeup brands (NATASHA DENONA, Charlotte Tilbury, NUDESTIX) where shade and formula variation across a large SKU catalog means the brand's aggregate health score is pulled down by a portion of its catalog that underperforms relative to the brand's halo products. For these brands, health score improvement is less a brand-level intervention than a SKU-level portfolio management problem: identifying the specific products generating mismatch and either reformulating, repricing, or resetting expectations for those products while protecting the high-health core.